In [8]:
#RNN 
import pandas as pd
import re
import nltk
import torch
import torch.nn as nn
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from torch.utils.data import TensorDataset,DataLoader
import torch.optim as optim

df=pd.read_csv("IMDB Dataset.csv")
# print(df.shape)
# print(df.isnull().sum()) #checking null 

df.drop_duplicates(inplace=True)
# print(df.shape)

#preprocessing
# 1. convert to lowercase,2.Remove Urls , remove punctautions,HTML Tags ,stop words(general words)
# 2.stemming
# 3.Encode Sentiment +vectorization

#lowercase
df["review"]=df["review"].str.lower()
# print(df.head())

#remove urls
def remove_urls(text):
    text=re.sub(r"http\S+"," ",text)
    return text
df["review"]=df["review"].apply(remove_urls)

#removing punctuations
def remove_punctuations(text):
    text=re.sub(r"[^A-Za-z0-9\s]"," ",text)
    return text
df["review"]=df["review"].apply(remove_punctuations)

# print(df.head())

#removing html
def remove_html(text):
    text=re.sub(r"<*?>"," ",text)
    return text
df["review"]=df["review"].apply(remove_html)

#removing stopwords
# nltk.download("punkt")
# nltk.download("punkt_tab")
# nltk.download("stopwords")

def remove_stopwords(text):
    tokens=word_tokenize(text)
    stop_words=stopwords.words("english")
    
    for word in tokens:
        if word in stop_words:
            text=text.replace(word," ")
            
    return text

df["review"]=df["review"].apply(remove_stopwords)

#Stemming
def stemming(text):
    ps=PorterStemmer()
    stemmed_words=[]
    tokens=word_tokenize(text)
    for token in tokens:
        stemmed_token=ps.stem(token)
        stemmed_words.append(stemmed_token)
        
    return " ".join(stemmed_words)
df["review"]=df["review"].apply(stemming)
# print(df.head())

#Encoding
le=LabelEncoder()
df["sentiment"]=le.fit_transform(df["sentiment"])
y=df["sentiment"]

#vectorization
tfidVec=TfidfVectorizer(max_features=5000)
X=tfidVec.fit_transform(df["review"])

# print(pd.DataFrame(X[:5].toarray()))
# print(X.shape)

#TensorDataSet and DataLoader
X_train,X_test,y_train,y_test=train_test_split(
    X,y,test_size=0.2,random_state=42
)

X_train=X_train.toarray()
X_test=X_test.toarray()

X_train=torch.tensor(X_train,dtype=torch.float32)
X_test=torch.tensor(X_test,dtype=torch.float32)
y_train=torch.tensor(y_train.values,dtype=torch.float32)
y_test=torch.tensor(y_test.values,dtype=torch.float32)


#Create TensorDataset
trainSet=TensorDataset(X_train,y_train)
testSet=TensorDataset(X_test,y_test)

#Loader
trainLoader=DataLoader(trainSet,batch_size=32,shuffle=True)
testLoader=DataLoader(testSet,batch_size=32,shuffle=True)


#Define RNN
class RNN(nn.Module):   #Many to 1 architecture for RNN 
    def __init__(self,input_size,hidden_size=128,num_layers=1):
        super().__init__()
        
        self.hidden_size=hidden_size
        self.num_layers=num_layers
        
        self.rnn=nn.RNN(input_size,hidden_size,num_layers,batch_first=True)
        
        self.fc=nn.Linear(hidden_size,1) 
        
        
    def forward(self,X):
        ho=torch.zeros(self.num_layers,X.size(0),self.hidden_size)
        
        out,_=self.rnn(X,ho)
        
        out=self.fc(out[:,-1,:])
        return out
    
#Creating Model
input_size=X_train.shape[1]
model=RNN(input_size)

criterion=nn.BCELoss()  
optimizer=optim.Adam(model.parameters())

#Training RNN
num_epochs=10

for epoch in range(num_epochs):
    model.train()
    
    for xb,yb in trainLoader:
        optimizer.zero_grad()
        xb=xb.unsqueeze(1)
        
        outputs=model(xb)
        outputs=torch.sigmoid(outputs.squeeze())
        
        loss=criterion(outputs,yb)
        loss.backward()
        optimizer.step()
        
    print(f"epochs={epoch+1}/{num_epochs} and loss={loss.item()}")
    

#Validation
model.eval()

with torch.no_grad():
    corr=0
    tot=0
    for xb,yb in testLoader:
        xb=xb.unsqueeze(1)
        
        outputs=model(xb)
        predicted=(torch.sigmoid(outputs.squeeze())>0.5).float()
        
        tot+=yb.size(0)
        corr+=(predicted==yb).sum().item()
        
    print(f"acc={corr/tot*100}")
        
        
    
        

epochs=1/10 and loss=0.4778488278388977
epochs=2/10 and loss=0.3040882647037506
epochs=3/10 and loss=0.2945132553577423
epochs=4/10 and loss=0.2933589220046997
epochs=5/10 and loss=0.5196499228477478
epochs=6/10 and loss=0.16359612345695496
epochs=7/10 and loss=0.3775973618030548
epochs=8/10 and loss=0.33225956559181213
epochs=9/10 and loss=0.22346842288970947
epochs=10/10 and loss=0.28826355934143066
acc=81.85943329635978
